In [1]:
import os
from dotenv import load_dotenv
import chromadb
from chromadb.config import Settings
from datasets import load_dataset, Dataset
from datasets import get_dataset_config_names

In [2]:
name_ds = "deepset/germanquad"
configs = get_dataset_config_names(name_ds)
print(configs)

demodata: Dataset = load_dataset(name_ds, split='test')  # type: ignore

print(demodata.num_rows)
print(demodata.shape) # Shape of the dataset (number of columns, number of rows).
print(demodata.column_names)
print(demodata.features)

['plain_text']
2204
(2204, 4)
['id', 'context', 'question', 'answers']
{'id': Value(dtype='int32', id=None), 'context': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'answers': Sequence(feature={'text': Value(dtype='string', id=None), 'answer_start': Value(dtype='int32', id=None)}, length=-1, id=None)}


In [3]:
load_dotenv()

chroma_client = chromadb.HttpClient(host='localhost', port="8000",settings=Settings(
            chroma_client_auth_provider="chromadb.auth.token.TokenAuthClientProvider",
            chroma_client_auth_credentials="test-token",
        ))

colls = chroma_client.list_collections()
colls

[Collection(name=testdata), Collection(name=documents)]

In [5]:
# Achtung: danach sind die Collection Connections weg => Server neu starten
# chroma_client.delete_collection('testdata')


collection = chroma_client.get_or_create_collection('testdata')

In [6]:
collection.count()
# collection.name

621

In [ ]:
collection.peek(100)


In [13]:
import json
# split in first 100 and rest
ab = 150
bis = 200 # dataset.num_rows

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

data = []
for i in range(bis-ab):
  question = {}
  question["id"] = f'Q_{slice["id"][i]}'
  # question["process_id"]= str(slice["id"][i])
  question["process_id"]= f"Slice_{ab}_{bis}"
  question["gdpr_id"]= str(slice["id"][i])
  question["topic"]= slice["question"][i]
  question["type"]= "question"
  question["len"] = len(slice["question"][i])
  question["document"]= slice["question"][i]
  data.append(question)

  answer = {}
  answer["id"] = f'A_{slice["id"][i]}'
  # answer["process_id"]= str(slice["id"][i])
  answer["process_id"]= f"Slice_{ab}_{bis}"
  answer["gdpr_id"]= str(slice["id"][i])
  answer["topic"]= slice["answers"][i]["text"][0]
  answer["type"]= "answer"
  answer["len"] = len(slice["context"][i])
  answer["document"]= slice["context"][i]
  data.append(answer)


In [14]:
len(data)
# data

100

In [18]:
data[0]["process_id"]

'Slice_0_50'

In [10]:
import os
import requests

api_url = "http://127.0.0.1:8080/embedding/insert/"

headers = {
  'Content-Type': 'application/json',
  'access_token': os.getenv("API_KEY")
}

response = requests.put(api_url, json=data, headers=headers)
if response.status_code != 200:
  print("Error at: ", response.json)

In [ ]:
print(response.json())
len(response.json())

In [12]:
import pandas as pd

df = pd.DataFrame(data)
print(df)

         id     process_id gdpr_id  \
0   Q_41225  Slice_100_150   41225   
1   A_41225  Slice_100_150   41225   
2   Q_41226  Slice_100_150   41226   
3   A_41226  Slice_100_150   41226   
4   Q_41228  Slice_100_150   41228   
..      ...            ...     ...   
95  A_38516  Slice_100_150   38516   
96  Q_38517  Slice_100_150   38517   
97  A_38517  Slice_100_150   38517   
98  Q_38518  Slice_100_150   38518   
99  A_38518  Slice_100_150   38518   

                                                topic      type  len  \
0   Was ist der Fachbegriff für die Gleichgewichts...  question   67   
1                           Equilibrium Line Altitude    answer  563   
2   Was wird als Gleichgewichtslinie eines Gletsch...  question   61   
3                                         Höhengrenze    answer  563   
4    Wie ist das Zehrgebiet des Gletschers definiert?  question   48   
..                                                ...       ...  ...   
95                                     